# Рынок заведений общественного питания Москвы

## Описание проекта.

* Инвесторы из фонда «Shut Up and Take My Money» решили попробовать себя в новой области и открыть заведение общественного питания в Москве. Заказчики ещё не знают, что это будет за место: кафе, ресторан, пиццерия, паб или бар, — и какими будут расположение, меню и цены.  
* Для начала они просят вас — аналитика — подготовить исследование рынка Москвы, найти интересные особенности и презентовать полученные результаты, которые в будущем помогут в выборе подходящего инвесторам места.  
* Постарайтесь сделать презентацию информативной и лаконичной. Её структура и оформление сильно влияют на восприятие информации читателями вашего исследования. Выбирать инструменты (matplotlib, seaborn и другие) и типы визуализаций вы можете самостоятельно.  
* Вам доступен датасет с заведениями общественного питания Москвы, составленный на основе данных сервисов Яндекс Карты и Яндекс Бизнес на лето 2022 года. Информация, размещённая в сервисе Яндекс Бизнес, могла быть добавлена пользователями или найдена в общедоступных источниках. Она носит исключительно справочный характер.  

## Описание данных.

Файл moscow_places.csv:
- name — название заведения;
- address — адрес заведения;
- category — категория заведения, например «кафе», «пиццерия» или «кофейня»;
- hours — информация о днях и часах работы;
- lat — широта географической точки, в которой находится заведение;
- lng — долгота географической точки, в которой находится заведение;
- rating — рейтинг заведения по оценкам пользователей в Яндекс Картах (высшая оценка — 5.0);
- price — категория цен в заведении, например «средние», «ниже среднего», «выше среднего» и так далее;
- avg_bill — строка, которая хранит среднюю стоимость заказа в виде диапазона, например:
    - «Средний счёт: 1000–1500 ₽»;
    - «Цена чашки капучино: 130–220 ₽»;
    - «Цена бокала пива: 400–600 ₽».
и так далее;
- middle_avg_bill — число с оценкой среднего чека, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Средний счёт»:
    - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    - Если значения нет или оно не начинается с подстроки «Средний счёт», то в столбец ничего не войдёт.
- middle_coffee_cup — число с оценкой одной чашки капучино, которое указано только для значений из столбца avg_bill, начинающихся с подстроки «Цена одной чашки капучино»:
    - Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    - Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    - Если значения нет или оно не начинается с подстроки «Цена одной чашки капучино», то в столбец ничего не войдёт.
- chain — число, выраженное 0 или 1, которое показывает, является ли заведение сетевым (для маленьких сетей могут встречаться ошибки):
    - 0 — заведение не является сетевым
    - 1 — заведение является сетевым
- district — административный район, в котором находится заведение, например Центральный административный округ;
- seats — количество посадочных мест.

## Импорт библиотек и изучение данных.

### Импорт библиотек

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly
import plotly.express as px
from plotly import graph_objects as go
import folium
from folium import Map, Choropleth, Marker
from folium.plugins import MarkerCluster
from folium.features import CustomIcon
import json
import warnings
warnings.filterwarnings("ignore")

### Загрузка данных

In [ ]:
path = 'C:/Users/Didid/Downloads/'
try:
    df = pd.read_csv(path + 'moscow_places.csv')
except:
    df = pd.read_csv('/datasets/moscow_places.csv')

### Изучаем данные

In [ ]:
display(df.head())

In [ ]:
display(df.info())

In [ ]:
display(df.describe().T)

### Переводим в нижний регистр значения 

In [ ]:
df['name'] = df['name'].str.lower()
df['address'] = df['address'].str.lower()
df['avg_bill'] = df['avg_bill'].str.lower()

In [ ]:
display(df['name'].nunique())

**Вывод:**
* Столбцы с названием, адресом и средней стоимостью переведены в нижний регист для удобства работы.
* Датасет имеет 5512 уникаленых заведений.
* Все столбцы имеют нужный тип данных. 

## Предобработка данных

### Явные дубликаты

In [ ]:
display(df.duplicated().sum())

### Не явные дубликаты

In [ ]:
display(df.duplicated(subset = ['name', 'address']).sum())

In [ ]:
df = df.drop_duplicates(subset = ['name', 'address'])

### Работа с пропусками

In [ ]:
display(df.isna().sum())

*Промежуточный вывод:*  
* avg_bill, hours, seats, price уникальные значения, заполнить их не получится.
* middle_avg_bill и middle_coffee_cup заполняются на основе avg_bill, их тоже заполнить нет возможности.  

Пропуски оставим как есть.

### Создание столбеца street с названиями улиц из столбца с адресом.

In [ ]:
df['street'] = df['address'].str.split(',').str[1]

In [ ]:
display(df['street'])

### Создание столбеца is_24/7 с обозначением, что заведение работает ежедневно и круглосуточно (24/7).

In [ ]:
df['is_24_7'] = df['hours'] == 'ежедневно, круглосуточно'

In [ ]:
display(df['is_24_7'].unique())

**Вывод:**
* Удалили дубликаты.
* Проанализировали пропуски и пришли к выводу, что убрать их нельзя.
* Создали два новых столбца street (названия улиц) и is_24_7 (работает ли заведение 24 часа или нет) для будущего удобства в анализе.

## Анализ данных

### Исследование количества объектов общественного питания по категориям: рестораны, кофейни, пиццерии, бары и так далее.

In [ ]:
category_count = df.pivot_table(index='category', values='name', aggfunc='count').sort_values(by='name', ascending=False).reset_index()

fig = px.bar(category_count, x='category', y='name',
             title='Распределение заведений по категориям',
             labels={'name': 'Количество заведений', 'category': 'Категория'})
fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

*Промежуточный вывод:*
* Больше всего в Москве кафе (2376), далее рестораны (2042), на третьем месте кофейни (1413).
* Меньше всего в Москве булочных (256), столовых (315)

### Исследование количества посадочных мест в местах по категориям: рестораны, кофейни, пиццерии, бары и так далее.

In [ ]:
seat_count = df.pivot_table(index = 'category', values = 'seats', aggfunc = 'median').sort_values(by = 'seats', ascending = False).reset_index()
fig = px.bar(seat_count, x='category', y='seats',
             title='Медианное количество мест в заведении по категориям',
             labels={'seats': 'Количество мест', 'category': 'Категория'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

*Промежуточные выводы:*
* Наибольшее медианное количество мест в ресторанах (86), барах и пабах (82), кофейнях (80).
* Наименьшее медианное количество мест в булочных (50), пиццериях (55).

### Исследование соотношения сетевых и несетевых заведений в датасете.

In [ ]:
df['chain'] = df['chain'].replace(0, 'несетевые')
df['chain'] = df['chain'].replace(1, 'сетевые')

In [ ]:
chain = df.pivot_table(index='chain', values='name', aggfunc='count').sort_values(by='name', ascending=False).reset_index()
chain.columns = ['chain', 'count']

fig = px.pie(chain, values='count', names='chain', hole=.3,
             title='Соотношение "сетевых" и "несетевых" заведений')
fig.show()

*Промежуточный вывод*  
Большая часть заведений не сетевые 5199 (61.9%), сетевых 3203 (38,1%).

### Какие категории заведений чаще являются сетевыми?

In [ ]:
category_count = df.pivot_table(index='category', values='name', aggfunc='count').sort_values(by='name', ascending=False).reset_index()
category_count.columns = ['category', 'count']

chain_category = df.query('chain == "сетевые"').pivot_table(index='category', values='name', aggfunc='count').sort_values(by='name', ascending=False).reset_index()
chain_category.columns = ['category', 'chain']

chain_category_merge = category_count.merge(chain_category, how='left', on='category').reset_index()
chain_category_merge.columns = ['index', 'category', 'count', 'chain']
chain_category_merge.drop('index', axis=1, inplace=True) 


chain_category_merge['percent'] = round(chain_category_merge['chain']/chain_category_merge['count']*100, 2)
chain_category_merge.fillna(0, inplace=True) 

fig = px.bar(chain_category_merge.sort_values(by='percent', ascending=False),
             x='category', y='percent',
             title='Процент "сетевых" заведений по категориям',
             labels={'category': 'Категория', 'percent': 'Процент "сетевых" заведений'})
fig.update_layout(xaxis_tickangle=-45) 
fig.show()

*Промежуточный вывод:*
* Чаще всего сетевыми являются булочные в 61,33%, пиццерии в 52,13%.
* Реже всего сетевыми являются бары и пабы в 21,99%, столовые в 27.94%.

### Топ-15 популярных сетей в Москве

In [ ]:
top_chain = df.query('chain == "сетевые"').groupby('name')['chain'].count().sort_values(ascending=False).head(15).reset_index()
top_chain.columns = ['name', 'count'] 

fig = px.bar(top_chain, x='name', y='count',
             title='Топ-15 сетевых заведений',
             labels={'name': 'Название заведения', 'count': 'Количество заведений в сети'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
top_chain_with_categories = df.query('chain == "сетевые"').groupby(['name', 'category'])['chain'].count().reset_index()
top_chain_with_categories.columns = ['name', 'category', 'count']
top_chain_categories = top_chain_with_categories[top_chain_with_categories['name'].isin(top_chain['name'])]

print("\nТаблица категорий для топ-15 сетевых заведений:")
print(top_chain_categories[['name', 'category', 'count']].to_string(index=False))

*Промежуточный вывод:*  
* В топ-3 сетей входят шоколадница (120 заведений), доминос пицца (76 заведений) и додо пицца (74 заведений). Замыкает топ-15 му-му (27 заведений).
* Анализируя категории сетей, можно прийти к выводу, что в одной сети может быть указано у разных заведений разные категории. Однако можно выделить основные. Например, кофейня у шоколадницы, додо и доминос это пиццерия, one price coffee - кофейня, му-му - кафе и так далее.

### Исследование общего количества заведений и количества заведений каждой категории по районам.

In [ ]:
district = df.pivot_table(index = 'district', values = 'name', aggfunc = 'count').sort_values(by = 'name', ascending = False).reset_index()
fig = px.bar(district, x='district', y='name',
             title='Количество заведений по административным округам',
             labels={'name': 'Количество заведений', 'district': 'Округ'})
fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

In [ ]:
district_2 = df.groupby(['district', 'category']).size().reset_index(name='count')

district_2 = district_2.sort_values(by='count', ascending=False)

fig = px.bar(district_2, x='district', y='count', color='category',
             title='Количество заведений по категориям по административным округам', 
             labels={'district': 'Административный округ', 'count': 'Количество заведений', 'category': 'Категория'}, 
             )
fig.update_layout(xaxis_tickangle=-45, height=800, width=1000, barmode = 'group') 
fig.show()

*Промежуточный вывод:*
* Больше всего заведений в ЦАО (2242), меньше всего в СЗАО (409).
* В ЦАО больше всего заведений категории ресторан, в остальных округах самая частая категория заведения кафе.

### Исследование распределения средних рейтингов по категориям заведений.

In [ ]:
ratings = df.groupby('category').agg(mean_rating=('rating', 'mean')).sort_values(by='mean_rating', ascending=False).reset_index()
fig = px.bar(ratings, x='category', y='mean_rating',
             title='Средний рейтинг по категориям заведений',
             labels={'mean_rating': 'Средний рейтинг', 'category': 'Категории'})
fig.update_layout(xaxis_tickangle=-45, height=500, width=700)
fig.show()

*Промежуточный вывод:*  
Разница в среднем рейтинге минимальна. Самый высокий средний рейтинг у баров и пабов 4.39, самый низкий средний рейтинг у заведений быстрого питания 4.05.

### Исследование фоновой картограммы (хороплет) со средним рейтингом заведений каждого района.

In [ ]:
district_ratings = df.groupby('district').agg(mean_rating=('rating', 'mean')).sort_values(by='mean_rating', ascending=False).reset_index()

In [ ]:
state_geo = 'admin_level_geomap.geojson'
    
moscow_lat, moscow_lng = 55.751244, 37.618423

m = Map(location=[moscow_lat, moscow_lng], zoom_start=9)

Choropleth(
    geo_data=state_geo,
    data=district_ratings,
    columns=['district', 'mean_rating'],
    key_on='feature.name',
    fill_color='YlGn',
    fill_opacity=0.5,
    legend_name='Средний рейтинг заведений по округам',
).add_to(m)

display(m)

In [ ]:
display(district_ratings)

*Промежуточный вывод:*  
* Самый высокий средний рейтинг заведения в ЦАО (4.38).
* Самый низкий средний рейтинг заведения в ЮВАО (4.10).

### Вывод всех заведений датасета на карте с помощью кластеров

In [ ]:
m = folium.Map(location=[moscow_lat, moscow_lng], zoom_start=10)

# Создаём кластер маркеров
marker_cluster = MarkerCluster().add_to(m)

# Итерируемся по строкам DataFrame и добавляем маркеры в кластер
for index, row in df.iterrows():
    folium.Marker(
        location=[row['lat'], row['lng']],
        popup=f"{row['name']} {row['rating']}",
    ).add_to(marker_cluster)

# Выводим карту
display(m)

*Промежуточный вывод:*  
Выведены все заведения при помощи кластеров. Больше всего заведений в центральных кластерах, что закономерно учитывая анализ выше.

### Топ-15 улиц по количеству заведений.

In [ ]:
top_street = (
    df.groupby('street')['name']
    .count()
    .sort_values(ascending=False)
    .head(15)
    .reset_index(name='count')
)

top_street.rename(columns={'count': 'name'}, inplace=True)

fig = px.bar(top_street, x='street', y='name',
             title='Топ-15 улиц по количеству заведений',
             labels={'name': 'количество', 'street': 'Улицы'})
fig.update_layout(xaxis_tickangle=-45, height=500, width=700)
fig.show()

In [ ]:
top_streets = top_street['street'].tolist()

street_per_category = df[df['street'].isin(top_streets)].groupby(['street', 'category']).size().reset_index(name='count')

fig = px.bar(street_per_category.sort_values(by='count', ascending=False),
             x='street',
             y='count',
             color='category',
             title='Количество заведений каждой категории по улицам',
             labels={'street': 'Улицы', 'count': 'Количество заведений'})

fig.update_layout(xaxis_tickangle=-45, height=800, width=1000, barmode = 'group')

fig.show()

*Промежуточный вывод:*  
* Первое место в топ-15 занимает проспект мира (183 заведения), последнее место в топ-15 занимает пятницкая улица (48 заведений).
* На данных улицах превалируют категории заведений Кафе и Ресторан.

### Исследование улиц, на которых находится только один объект общепита.

In [ ]:
street_one_place = (
    df.groupby('street')['name']
    .count()
    .reset_index(name='count')
)

street_one_place = street_one_place[street_one_place['count'] == 1]

onlyone_streets = street_one_place['street'].tolist()

df_onlyone_street = df[df['street'].isin(onlyone_streets)]

In [ ]:
display(df_onlyone_street.head())

In [ ]:
display(df_onlyone_street.info())

In [ ]:
d_1 = df_onlyone_street.groupby('category')['name'].count().sort_values(ascending=False).reset_index(name='count')

d_1.rename(columns={'count': 'name'}, inplace=True)

fig = px.bar(d_1, x='category', y='name',
             title='Распределение рассматриваемых заведений по категориям',
             labels={'name': 'количество', 'category': 'Категории'})

fig.update_layout(xaxis_tickangle=-45, height=500, width=700)
fig.show()

In [ ]:
d_2 = df_onlyone_street.groupby('district')['name'].count().sort_values(ascending=False).reset_index(name='count')

d_2.rename(columns={'count': 'name'}, inplace=True)

fig = px.bar(d_2, x='district', y='name',
             title='Распределение рассматриваемых заведений по округам',
             labels={'name': 'количество', 'district': 'Округ'})

fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

In [ ]:
chain = df_onlyone_street.pivot_table(index='chain', values='name', aggfunc='count').sort_values(by='name', ascending=False).reset_index()
chain.columns = ['chain', 'count']

fig = px.pie(chain, values='count', names='chain', hole=.3,
             title='Соотношение "сетевых" и "несетевых" заведений')
fig.show()

*Промежуточный вывод:*
* Всего улиц на которых одно заведение 457.
* Чаще всего это кафе (159), реже всего это булочные (8).
* Больше всего таких улиц в ЦАО, меньше всего в ЮЗАО.
* Большая часть это несетевые заведения 70.9% (324).

### Посчитайте медиану middle_avg_bill для каждого района. Используйте это значение в качестве ценового индикатора района. Постройте фоновую картограмму (хороплет) с полученными значениями для каждого района.

In [ ]:
d_m_m_a_b = (
    df.groupby('district')['middle_avg_bill']
    .median()
    .sort_values(ascending=False)
    .reset_index(name='median_middle_avg_bill')
)

In [ ]:
m = Map(location=[moscow_lat, moscow_lng], zoom_start=10)

Choropleth(
    geo_data=state_geo,
    data=d_m_m_a_b,
    columns=['district', 'median_middle_avg_bill'],
    key_on='feature.name',
    fill_color='YlGn',
    fill_opacity=0.8,
    legend_name='Медианный средний чек заведений по округам',
).add_to(m)

display(m)

In [ ]:
display(d_m_m_a_b)

*Промежуточный вывод:*  
* Самые высокие средние чеки в ЦАО и ЗАО (1000 рублей). Самые низкие в ЮВАО (450 рублей).
* Если анализировать влияет ли удалённость от центра на средние чеки, то по этим данным сложно судить. Скорее всего деление на районы поможет ответить на этот вопрос, так как при делении по округам берутся слишком большие территории.

**Общий вывод:**  
* Больше всего в Москве кафе (2376), далее рестораны (2042), на третьем месте кофейни (1413).
* Меньше всего в Москве булочных (256), столовых (315).
* Наибольшее медианное количество мест в ресторанах (86), барах и пабах (82), кофейнях (80).
* Наименьшее медианное количество мест в булочных (50), пиццериях (55).
* Большая часть заведений Москвы не сетевые 5199 (61.9%), сетевых 3203 (38,1%).
* Чаще всего сетевыми в Москве являются булочные в 61,33%, пиццерии в 52,13%.
* Реже всего сетевыми в Москве являются бары и пабы в 21,99%, столовые в 27.94%.
* В топ-3 сетей входят шоколадница (120 заведений), доминос пицца (76 заведений) и додо пицца (74 заведений). Замыкает топ-15 му-му (27 заведений).
* Анализируя категории сетей, можно прийти к выводу, что в одной сети может быть указано у разных заведений разные категории. Однако можно выделить основные. Например, кофейня у шоколадницы, додо и доминос это пиццерия, one price coffee - кофейня, му-му - кафе и так далее.
* Больше всего заведений в ЦАО (2242), меньше всего в СЗАО (409).
* В ЦАО больше всего заведений категории ресторан, в остальных округах самая частая категория заведения кафе.
* Разница в среднем рейтинге минимальна. Самый высокий средний рейтинг у баров и пабов 4.39, самый низкий средний рейтинг у заведений быстрого питания 4.05.
* Самый высокий средний рейтинг заведения в ЦАО (4.38).
* Самый низкий средний рейтинг заведения в ЮВАО (4.10).
* Выведены все заведения при помощи кластеров. Больше всего заведений в центральных кластерах, что закономерно учитывая анализ выше.
* Первое место в топ-15 улиц по количеству заведений занимает проспект мира (183 заведения), последнее место в топ-15 занимает пятницкая улица (48 заведений).
* В топ-15 улиц по количеству заведений превалируют категории заведений Кафе и Ресторан.
* Всего улиц на которых одно заведение 457.
  - Чаще всего это кафе (159), реже всего это булочные (8).
  - Больше всего таких улиц в ЦАО, меньше всего в ЮЗАО.
  - Большая часть это несетевые заведения 70.9% (324).
* Самые высокие средние чеки в ЦАО и ЗАО (1000 рублей). Самые низкие в ЮВАО (450 рублей).
* Если анализировать влияет ли удалённость от центра на средние чеки, то по этим данным сложно судить. Скорее всего деление на районы поможет ответить на этот вопрос, так как при делении по округам берутся слишком большие территории.

## Детализируем исследование: открытие кофейни

In [ ]:
df_1 = df.query("category == 'кофейня'")
display(df_1)

*Промежуточный вывод*  
Всего 1413 кофеен в датасете.

In [ ]:
district = df_1.pivot_table(index = 'district', values = 'name', aggfunc = 'count').sort_values(by = 'name', ascending = False).reset_index()
fig = px.bar(district, x='district', y='name',
             title='Количество кофеен по административным округам',
             labels={'name': 'Количество кофеен', 'district': 'Округ'})
fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

*Промежуточный вывод:*  
Больше всего кофеен в ЦАО (428), меньше всего в СЗАО (62). Большая часть кофеен находится в центре около 30%. Остальные округа имеют от 4% до 14% кофеен от общего числа.

In [ ]:
m = Map(location=[moscow_lat, moscow_lng], zoom_start=10)

Choropleth(
    geo_data=state_geo,
    data=district,
    columns=['district', 'name'],
    key_on='feature.name',
    fill_color='YlGn',
    fill_opacity=0.8,
    legend_name='Количество кофеен по округам',
).add_to(m)

display(m)

На карте более показательно выглядит.

In [ ]:
df_1['is_24_7'] = df_1['is_24_7'].replace({True: 'круглосуточные 24/7', False: 'не круглосуточные'})

is_24_7 = (
    df_1.groupby('is_24_7')['name']
    .count()
    .sort_values(ascending=False)
    .reset_index(name='count')
)


fig = go.Figure(data=[go.Pie(labels=is_24_7['is_24_7'], values=is_24_7['count'])])
fig.update_layout(title='Соотношение "круглосуточных" и "не круглосуточных" заведений')
fig.show()

*Промежуточный вывод:*  
Абсолютное большинство кофеен не круглосуточные 95,8%.

In [ ]:
ratings = df_1.groupby('district').agg(mean_rating=('rating', 'mean')).sort_values(by='mean_rating', ascending=False).reset_index()
fig = px.bar(ratings, x='district', y='mean_rating',
             title='Средний рейтинг кофеен по Округам',
             labels={'mean_rating': 'Средний рейтинг', 'district': 'Округ'})
fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

In [ ]:
m = Map(location=[moscow_lat, moscow_lng], zoom_start=10)

Choropleth(
    geo_data=state_geo,
    data=ratings,
    columns=['district', 'mean_rating'],
    key_on='feature.name',
    fill_color='YlGn',
    fill_opacity=0.8,
    legend_name='Средний рейтинг кофеен по Округам',
).add_to(m)

display(m)

*Промежуточный вывод:*  
Самый высокий средний рейтинг у кофеен в ЦАО (4.34), самый низкий средний рейтинг у кофеен в ЗАО (4.20).

In [ ]:
middle_cup = df_1.groupby('district').agg(median_cost=('middle_coffee_cup', 'median')).sort_values(by='median_cost', ascending=False).reset_index()
fig = px.bar(middle_cup, x='district', y='median_cost',
             title='Медианная стоимость чашки кофе по Округам',
             labels={'median_cost': 'Медианная стоимость', 'district': 'Округ'})
fig.update_layout(xaxis_tickangle=-45, height=700, width=700)
fig.show()

In [ ]:
m = Map(location=[moscow_lat, moscow_lng], zoom_start=10)

Choropleth(
    geo_data=state_geo,
    data=middle_cup,
    columns=['district', 'median_cost'],
    key_on='feature.name',
    fill_color='YlGn',
    fill_opacity=0.8,
    legend_name='Медианная стоимость чашки кофе по Округам',
).add_to(m)

display(m)

*Промежуточный вывод:*  
Самая высокая медианная стоимость чашки кофе в ЮЗАО (198 рублей), самая низкая медианная стоимость чашки кофе в ВАО (135 рублей).

In [ ]:
moscow_lat, moscow_lng = 55.608740, 37.522436

m = folium.Map(location=[moscow_lat, moscow_lng], zoom_start=12)

# Создаём кластер маркеров
marker_cluster = MarkerCluster().add_to(m)

# Итерируемся по строкам DataFrame и добавляем маркеры в кластер
for index, row in df_1.iterrows():
    folium.Marker(
        location=[row['lat'], row['lng']],
        popup=f"{row['name']} {row['rating']}",
    ).add_to(marker_cluster)

# Выводим карту
display(m)

**Рекомендация:**  
На основе полученных данных самыми благоприятными округами для открытия кофеен являются ЗАО и ЮЗАО. Так как в них не самое большое количество кофеен, а в ЮЗАО одно из самых малых количеств. Однако, при этом в данных в данных округах очень высокая медианная стоимость чашек кофе, которая способствует возможности демпинга цен и овладеванием части рынка кофеен. Например, подходящее место есть в районе станций метро Тёплый стан и Ясенево, где всего 7 кофеен (см. на карту выше текста). Нахождение метро рядом даст большой поток клиентов. Полагаясь на данные с сайта о пассажиропотоке (<https://metrostat.ru/ru/moscow/2023-1>), где Тёплый Стан (108 тысяч человек в день) и Ясенево (56 тысяч человек в день), что является достаточно большим пассажиропотоком в сравнении с другими станциями метро, можно предположить присутсвуе дифицита заведений по продаже кофе в данном районе. 

***Общий вывод:***
1. Первая часть:  
* Столбцы с названием, адресом и средней стоимостью переведены в нижний регист для удобства работы.
* Датасет имеет 5512 уникаленых заведений.
* Все столбцы имеют нужный тип данных. 

2. Вторая часть:  
* Удалили дубликаты.
* Проанализировали пропуски и пришли к выводу, что убрать их нельзя.
* Создали два новых столбца street (названия улиц) и is_24_7 (работает ли заведение 24 часа или нет) для будущего удобства в анализе.

3. Третья часть:  
* Больше всего в Москве кафе (2376), далее рестораны (2042), на третьем месте кофейни (1413).
* Меньше всего в Москве булочных (256), столовых (315).
* Наибольшее медианное количество мест в ресторанах (86), барах и пабах (82), кофейнях (80).
* Наименьшее медианное количество мест в булочных (50), пиццериях (55).
* Большая часть заведений Москвы не сетевые 5199 (61.9%), сетевых 3203 (38,1%).
* Чаще всего сетевыми в Москве являются булочные в 61,33%, пиццерии в 52,13%.
* Реже всего сетевыми в Москве являются бары и пабы в 21,99%, столовые в 27.94%.
* В топ-3 сетей входят шоколадница (120 заведений), доминос пицца (76 заведений) и додо пицца (74 заведений). Замыкает топ-15 му-му (27 заведений).
* Анализируя категории сетей, можно прийти к выводу, что в одной сети может быть указано у разных заведений разные категории. Однако можно выделить основные. Например, кофейня у шоколадницы, додо и доминос это пиццерия, one price coffee - кофейня, му-му - кафе и так далее.
* Больше всего заведений в ЦАО (2242), меньше всего в СЗАО (409).
* В ЦАО больше всего заведений категории ресторан, в остальных округах самая частая категория заведения кафе.
* Разница в среднем рейтинге минимальна. Самый высокий средний рейтинг у баров и пабов 4.39, самый низкий средний рейтинг у заведений быстрого питания 4.05.
* Самый высокий средний рейтинг заведения в ЦАО (4.38).
* Самый низкий средний рейтинг заведения в ЮВАО (4.10).
* Выведены все заведения при помощи кластеров. Больше всего заведений в центральных кластерах, что закономерно учитывая анализ выше.
* Первое место в топ-15 улиц по количеству заведений занимает проспект мира (183 заведения), последнее место в топ-15 занимает пятницкая улица (48 заведений).
* В топ-15 улиц по количеству заведений превалируют категории заведений Кафе и Ресторан.
* Всего улиц на которых одно заведение 457.
  - Чаще всего это кафе (159), реже всего это булочные (8).
  - Больше всего таких улиц в ЦАО, меньше всего в ЮЗАО.
  - Большая часть это несетевые заведения 70.9% (324).
* Самые высокие средние чеки в ЦАО и ЗАО (1000 рублей). Самые низкие в ЮВАО (450 рублей).
* Если анализировать влияет ли удалённость от центра на средние чеки, то по этим данным сложно судить. Скорее всего деление на районы поможет ответить на этот вопрос, так как при делении по округам берутся слишком большие территории.

4. Четвёртая часть:  
* Всего 1413 кофеен в датасете.
* Больше всего кофеен в ЦАО (428), меньше всего в СЗАО (62).
* Абсолютное большинство кофеен не круглосуточные 95,8%.
* Самый высокий средний рейтинг у кофеен в ЦАО (4.34), самый низкий средний рейтинг у кофеен в ЗАО (4.20).
* Самая высокая медианная стоимость чашки кофе в ЮЗАО (198 рублей), самая низкая медианная стоимость чашки кофе в ВАО (135 рублей).

**Рекомендация по месту открытия кофейни:**  
На основе полученных данных самыми благоприятными округами для открытия кофеен являются ЗАО и ЮЗАО. Так как в них не самое большое количество кофеен, а в ЮЗАО одно из самых малых количеств. Однако, при этом в данных в данных округах очень высокая медианная стоимость чашек кофе, которая способствует возможности демпинга цен и овладеванием части рынка кофеен. Например, подходящее место есть в районе станций метро Тёплый стан и Ясенево, где всего 7 кофеен (см. на карту выше текста). Полагаясь на данные с сайта о пассажиропотоке (<https://metrostat.ru/ru/moscow/2023-1>), где Тёплый Стан (108 тысяч человек в день) и Ясенево (56 тысяч человек в день), что является достаточно большим пассажиропотоком в сравнении с другими станциями метро, можно предположить присутсвуе дифицита заведений по продаже кофе в данном районе.

Презентация: <https://drive.google.com/file/d/1u02F4VUn-pATHbSNcaBbc8RnlQM4m9GP/view?usp=sharing>

Выполнил Лосевский Дмитрий